# SIMPLIcity Pretrained CNN Architecture Benchmark Suite
### Course: Neuro Computing (PENS) - Case Study: Pretrained Vision Backbones on SIMPLIcity Dataset

This notebook provides an **end-to-end reproducible benchmarking pipeline** for comparing modern pretrained Computer Vision architectures on the **SIMPLIcity** dataset (1,000 images, 10 classes).

**Architectures Evaluated:**
1. **ResNet18** (Residual baseline, 11.7M parameters)
2. **ResNet34** (Deeper residual network, 21.8M parameters)
3. **MobileNetV3-Large** (Inverted residual + Squeeze-and-Excitation attention, 5.4M parameters)
4. **EfficientNet-B0** (Compound scaling depth/width/resolution, 5.3M parameters)
5. **ConvNeXt-Tiny** (Modernized pure ConvNet with 7x7 depthwise convolutions, 28.6M parameters)

**Key Methodological Rigor:**
- **Native Resolution**: 224x224 pixels matching ImageNet pretraining.
- **Differential Learning Rate**: Backbone fine-tunes with 10x lower learning rate (lr * 0.1) to preserve features.
- **Aspect-Ratio Preserved Transforms**: Scale cropping and center-cropping prevent distortion of landscape/portrait images.
- **Hardware Acceleration**: Automatic **Dual GPU Tesla T4 (T4 x2)** multi-GPU scaling via `DataParallel` + **FP16 AMP**.
- **Metrics**: Macro-F1, Accuracy, Confusion Matrix, and JSON Misclassification Error Analysis.


## 1. Verify GPU Acceleration & Dual T4 Topology
Checks GPU availability, device names, and VRAM. On Kaggle with **GPU T4 x2**, both GPUs will be detected.


In [ ]:
!nvidia-smi

import os

import torch

print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    count = torch.cuda.device_count()
    print(f"CUDA Device Count: {count}")
    for i in range(count):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_mem / (1024 ** 3)
        print(f"  GPU {i}: {name} ({mem:.1f} GB VRAM)")
    if count > 1:
        print("\n>>> Dual GPU T4 x2 detected! Multi-GPU DataParallel training is supported. <<<")
else:
    print("WARNING: No GPU detected. Please enable GPU accelerator in Kaggle/Colab runtime settings.")


## 2. Setup Environment & Repository
Clones or updates the repository, installs the package in editable mode (`pip install -e .`), and configures `sys.path`.


In [ ]:
import sys
from pathlib import Path

# 1. Ensure working directory is project root
if os.path.exists("src/simplicity_cnn"):
    print("Already inside cnn-SIMPLIcity project root.")
elif os.path.exists("../src/simplicity_cnn"):
    print("Currently in notebooks/ subdirectory. Moving to project root...")
    %cd ..
elif os.path.exists("cnn-SIMPLIcity/src/simplicity_cnn"):
    print("cnn-SIMPLIcity folder found. Entering project directory...")
    %cd cnn-SIMPLIcity
else:
    print("Cloning latest repository from GitHub...")
    !git clone -b main https://github.com/Arfiadi/cnn-SIMPLIcity.git
    %cd cnn-SIMPLIcity

# 2. Add 'src' directory directly to Python's sys.path
src_path = str(Path("src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    print(f"Added '{src_path}' to sys.path.")

# 3. Install project dependencies in editable mode
%pip install -q -e .
%pip install -q -r requirements.txt


## 3. Dataset Auto-Detection & Extraction
Automatically scans `/kaggle/input/**` (following symlinks and folder casing) or extracts `simplicity_dataset.zip` on Colab/Local.


In [ ]:
import os
import sys
import zipfile
from pathlib import Path

# Self-healing: Ensure 'src' is in sys.path and working directory is project root
for p in [Path("src"), Path("cnn-SIMPLIcity/src"), Path("/kaggle/working/cnn-SIMPLIcity/src"), Path("/content/cnn-SIMPLIcity/src")]:
    if (p / "simplicity_cnn").exists():
        p_resolved = str(p.resolve())
        if p_resolved not in sys.path:
            sys.path.insert(0, p_resolved)
        repo_candidate = p.parent
        if Path.cwd() != repo_candidate and (repo_candidate / "src").exists():
            os.chdir(repo_candidate)
        break

from simplicity_cnn.utils.env import get_default_data_dir  # noqa: E402

data_dir = get_default_data_dir()
train_exists = (data_dir / "train").exists() or (data_dir / "Train").exists()

if train_exists:
    print(f"[SUCCESS] Dataset detected and ready at: {data_dir}")
    subfolders = [p.name for p in data_dir.iterdir() if p.is_dir()]
    print(f"  Detected subfolders: {subfolders}")
else:
    zip_path = "simplicity_dataset.zip"
    target_dir = "Dataset/SIMPLIcity"
    if os.path.exists(zip_path):
        print(f"Extracting {zip_path} to {target_dir}...")
        os.makedirs(target_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(target_dir)
        data_dir = Path(target_dir).resolve()
        print(f"[SUCCESS] Extraction complete! Dataset ready at: {data_dir}")
    else:
        print(f"[NOTICE] Dataset 'train/' folder not found at: {data_dir}")
        if os.path.exists("/kaggle/input"):
            print("\nInspecting /kaggle/input contents:")
            for root, dirs, _ in os.walk("/kaggle/input", followlinks=True):
                print(f"  Folder: {root}")


## 4. Run Dataset Audit
Verifies image integrity and class distribution across train (500), validation (300), and test (200) splits.


In [ ]:
!python scripts/audit_data.py


---
## 5. Experiment: ResNet18 (Residual Baseline)
**Architecture Overview**: Standard 18-layer residual network (11.7M params). Serves as our primary transfer learning anchor.
- Input Resolution: $224 \times 224$ (ImageNet native)
- Optimizer: AdamW + Cosine Annealing (differential LR: backbone 10x lower)
- Multi-GPU: Enabled via `--multi-gpu` flag


In [ ]:
# Display Full Configuration for ResNet18 (Residual Baseline)
from simplicity_cnn.config.schema import print_config_summary  # noqa: E402

print_config_summary("configs/resnet18_transfer.yaml")


In [ ]:
# Execute Training for ResNet18 (Residual Baseline)
!python scripts/train.py \
    --config configs/resnet18_transfer.yaml \
    --multi-gpu \
    --backbone-lr-ratio 0.1


In [ ]:
# Display Training Curves & Confusion Matrix
from IPython.display import Image, display

print("=== ResNet18 (Residual Baseline) Gradient Descent Curves ===")
display(Image("outputs/resnet18_pretrained_224/gradient_descent_summary.png"))

print("=== ResNet18 (Residual Baseline) Confusion Matrix ===")
display(Image("outputs/resnet18_pretrained_224/evaluation/confusion_matrix.png"))


---
## 6. Experiment: ResNet34 (Deeper Residual Architecture)
**Architecture Overview**: 34-layer residual network (21.8M params). Tests whether doubling residual depth improves feature representation on subtle classes.
- Input Resolution: $224 \times 224$ (ImageNet native)
- Optimizer: AdamW + Cosine Annealing (differential LR: backbone 10x lower)
- Multi-GPU: Enabled via `--multi-gpu` flag


In [ ]:
# Display Full Configuration for ResNet34 (Deeper Residual Architecture)
from simplicity_cnn.config.schema import print_config_summary  # noqa: E402

print_config_summary("configs/resnet34_transfer.yaml")


In [ ]:
# Execute Training for ResNet34 (Deeper Residual Architecture)
!python scripts/train.py \
    --config configs/resnet34_transfer.yaml \
    --multi-gpu \
    --backbone-lr-ratio 0.1


In [ ]:
# Display Training Curves & Confusion Matrix
from IPython.display import Image, display

print("=== ResNet34 (Deeper Residual Architecture) Gradient Descent Curves ===")
display(Image("outputs/resnet34_pretrained_224/gradient_descent_summary.png"))

print("=== ResNet34 (Deeper Residual Architecture) Confusion Matrix ===")
display(Image("outputs/resnet34_pretrained_224/evaluation/confusion_matrix.png"))


---
## 7. Experiment: MobileNetV3-Large (Inverted Residuals + SE)
**Architecture Overview**: Efficient mobile architecture (5.4M params) with hard-swish non-linearities and Squeeze-and-Excitation attention blocks.
- Input Resolution: $224 \times 224$ (ImageNet native)
- Optimizer: AdamW + Cosine Annealing (differential LR: backbone 10x lower)
- Multi-GPU: Enabled via `--multi-gpu` flag


In [ ]:
# Display Full Configuration for MobileNetV3-Large (Inverted Residuals + SE)
from simplicity_cnn.config.schema import print_config_summary  # noqa: E402

print_config_summary("configs/mobilenet_v3_large.yaml")


In [ ]:
# Execute Training for MobileNetV3-Large (Inverted Residuals + SE)
!python scripts/train.py \
    --config configs/mobilenet_v3_large.yaml \
    --multi-gpu \
    --backbone-lr-ratio 0.1


In [ ]:
# Display Training Curves & Confusion Matrix
from IPython.display import Image, display

print("=== MobileNetV3-Large (Inverted Residuals + SE) Gradient Descent Curves ===")
display(Image("outputs/mobilenet_v3_large_pretrained_224/gradient_descent_summary.png"))

print("=== MobileNetV3-Large (Inverted Residuals + SE) Confusion Matrix ===")
display(Image("outputs/mobilenet_v3_large_pretrained_224/evaluation/confusion_matrix.png"))


---
## 8. Experiment: EfficientNet-B0 (Compound Scaling)
**Architecture Overview**: Principled depth/width/resolution compound scaling (5.3M params) with MBConv blocks and channel-wise recalibration.
- Input Resolution: $224 \times 224$ (ImageNet native)
- Optimizer: AdamW + Cosine Annealing (differential LR: backbone 10x lower)
- Multi-GPU: Enabled via `--multi-gpu` flag


In [ ]:
# Display Full Configuration for EfficientNet-B0 (Compound Scaling)
from simplicity_cnn.config.schema import print_config_summary  # noqa: E402

print_config_summary("configs/efficientnet_b0.yaml")


In [ ]:
# Execute Training for EfficientNet-B0 (Compound Scaling)
!python scripts/train.py \
    --config configs/efficientnet_b0.yaml \
    --multi-gpu \
    --backbone-lr-ratio 0.1


In [ ]:
# Display Training Curves & Confusion Matrix
from IPython.display import Image, display

print("=== EfficientNet-B0 (Compound Scaling) Gradient Descent Curves ===")
display(Image("outputs/efficientnet_b0_pretrained_224/gradient_descent_summary.png"))

print("=== EfficientNet-B0 (Compound Scaling) Confusion Matrix ===")
display(Image("outputs/efficientnet_b0_pretrained_224/evaluation/confusion_matrix.png"))


---
## 9. Experiment: ConvNeXt-Tiny (Modernized Pure ConvNet)
**Architecture Overview**: Modern CNN architecture (28.6M params) incorporating 7x7 depthwise convolutions, inverted bottlenecks, and LayerNorm.
- Input Resolution: $224 \times 224$ (ImageNet native)
- Optimizer: AdamW + Cosine Annealing (differential LR: backbone 10x lower)
- Multi-GPU: Enabled via `--multi-gpu` flag


In [ ]:
# Display Full Configuration for ConvNeXt-Tiny (Modernized Pure ConvNet)
from simplicity_cnn.config.schema import print_config_summary  # noqa: E402

print_config_summary("configs/convnext_tiny.yaml")


In [ ]:
# Execute Training for ConvNeXt-Tiny (Modernized Pure ConvNet)
!python scripts/train.py \
    --config configs/convnext_tiny.yaml \
    --multi-gpu \
    --backbone-lr-ratio 0.1


In [ ]:
# Display Training Curves & Confusion Matrix
from IPython.display import Image, display

print("=== ConvNeXt-Tiny (Modernized Pure ConvNet) Gradient Descent Curves ===")
display(Image("outputs/convnext_tiny_pretrained_224/gradient_descent_summary.png"))

print("=== ConvNeXt-Tiny (Modernized Pure ConvNet) Confusion Matrix ===")
display(Image("outputs/convnext_tiny_pretrained_224/evaluation/confusion_matrix.png"))


---
## 10. Comparative Results Summary & Decision Matrix
Consolidates performance metrics across all 5 pretrained architectures: Test Accuracy, Macro-F1, Parameter Count, and Training Time.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

experiments = [
    ("ResNet18", "outputs/resnet18_pretrained_224"),
    ("ResNet34", "outputs/resnet34_pretrained_224"),
    ("MobileNetV3-Large", "outputs/mobilenet_v3_large_pretrained_224"),
    ("EfficientNet-B0", "outputs/efficientnet_b0_pretrained_224"),
    ("ConvNeXt-Tiny", "outputs/convnext_tiny_pretrained_224"),
]

results = []
print("=" * 75)
print(f"{'Model Architecture':<20} {'Test Acc (%)':>14} {'Macro F1':>12} {'Best Epoch':>12}")
print("=" * 75)

for name, out_dir in experiments:
    report_file = Path(out_dir) / "evaluation" / "classification_report.json"
    hist_file = Path(out_dir) / "history.json"
    if report_file.exists():
        with open(report_file, "r", encoding="utf-8") as f:
            rep = json.load(f)
        with open(hist_file, "r", encoding="utf-8") as f:
            hist = json.load(f)
        acc = rep.get("accuracy", 0.0) * 100.0
        f1 = rep.get("macro avg", {}).get("f1-score", 0.0)
        best_ep = hist["val_macro_f1"].index(max(hist["val_macro_f1"])) + 1
        results.append((name, acc, f1, best_ep))
        print(f"{name:<20} {acc:>13.2f}% {f1:>12.4f} {best_ep:>12}")
    else:
        print(f"{name:<20} {'(not run yet)':>14} {'---':>12} {'---':>12}")
print("=" * 75)

# Plot comparative bar charts if results exist
if results:
    names = [r[0] for r in results]
    accs = [r[1] for r in results]
    f1s = [r[2] for r in results]

    fig, ax1 = plt.subplots(figsize=(10, 5))
    x = range(len(names))
    ax1.bar(x, accs, color='#1f77b4', width=0.4, label='Test Accuracy (%)')
    ax1.set_ylabel('Accuracy (%)', color='#1f77b4', fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(names, rotation=15, ha='right')
    ax1.set_ylim(80, 100)
    for i, v in enumerate(accs):
        ax1.text(i, v + 0.5, f"{v:.2f}%", ha='center', fontweight='bold', fontsize=9)

    ax2 = ax1.twinx()
    ax2.plot(x, f1s, color='#d62728', marker='o', linewidth=2.5, label='Macro F1')
    ax2.set_ylabel('Macro F1', color='#d62728', fontweight='bold')
    ax2.set_ylim(0.80, 1.0)
    for i, v in enumerate(f1s):
        ax2.text(i, v + 0.005, f"{v:.4f}", ha='center', color='#d62728', fontweight='bold', fontsize=9)

    plt.title('SIMPLIcity Pretrained Architecture Comparison', fontsize=13, fontweight='bold')
    fig.tight_layout()
    plt.show()


---
## 11. Error Analysis & Misclassification Inspection
Loads `misclassified_samples.json` across evaluated models to assess if the 224x224 native resolution and modern backbones eliminated errors on `People` and `Beach` vs `Mountain`.


In [ ]:
import json
from pathlib import Path

for name, out_dir in experiments:
    misclass_file = Path(out_dir) / "evaluation" / "misclassified_samples.json"
    if misclass_file.exists():
        with open(misclass_file, "r", encoding="utf-8") as f:
            errors = json.load(f)
        print(f"\n=== {name}: Total Misclassified: {len(errors)} / 200 ===")
        for err in errors[:5]:
            print(f"  File: {err['filename']:<12} | True: {err['true_label']:<10} -> Pred: {err['predicted_label']:<10} (Conf: {err['confidence']:.4f})")
    else:
        print(f"\n=== {name}: No evaluation output found yet ===")
